# Experiment: stylometry
Reference: TF-IDF baseline. Same 50k subsample for vocabulary / grid and same 100k training sample as the baseline (same sampling, same seed).
1. stylometry only (HistGradientBoosting)
2. TF-IDF + stylometry (baseline linear model with scaled stylometric features appended)

In [1]:
import gc
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
LABELS = ['human', 'ai', 'fusion']
SUB_N = 50_000
FULL_N = 100_000
INPUT = Path('/kaggle/input')
train = pd.read_parquet(next(INPUT.rglob('train_clean.parquet')), columns=['id', 'text', 'lang', 'domain', 'label', 'fold'])
test = pd.read_parquet(next(INPUT.rglob('test_clean.parquet')))

def strat_sample(df, n):
    return df.groupby(['label', 'lang', 'domain'], group_keys=False).sample(frac=min(n / len(df), 1), random_state=SEED)

tr, va = train[train.fold == 'train'], train[train.fold == 'val']
del train
sub_tr = strat_sample(tr, SUB_N)
tr = strat_sample(tr, FULL_N)

best_path = next(INPUT.rglob('baseline_best.json'), None)
BASE = json.load(open(best_path)) if best_path else {'model': 'svc', 'C': .3, 'class_weight': 'None', 'feats': 'word+char'}
len(tr), len(va), len(sub_tr), BASE

(99998,
 54537,
 50000,
 {'model': 'svc', 'C': 0.3, 'class_weight': 'None', 'feats': 'word+char'})

## Stylometric features
Global: lengths, word / sentence length, burstiness (sentence length CV), TTR, punctuation, case, digits, AI markers per 1k words.
Positional (4 quarters of the text): density of tokens the LLM adds / removes (from the human↔fusion pair diff), std across quarters and tail − head. Fusion keeps the human beginning and rewrites the rest.

In [2]:
SENT = re.compile(r'(?<=[.!?])\s+')
MARKERS = {
    'em_dash': re.compile('—'),
    'en_dash_spaced': re.compile(' – '),
    'curly_quotes': re.compile('[“”‘’„]'),
    'ellipsis_char': re.compile('…'),
    'markdown': re.compile(r'\*\*|^#|\s#{1,3}\s'),
    'title_colon_start': re.compile(r'^[^.!?]{10,120}:\s'),
    'placeholder': re.compile(r'\[[A-Za-z][^\]]{2,40}\]'),
    'not_x_but_y': re.compile(r"(?i)\b(?:not (?:just|only|merely|simply)\b[^.]{0,80}\bbut|isn't (?:just )?about|it's not\b[^.]{0,60}\bit's)\b"),
    'nicht_nur_sondern': re.compile(r'(?i)\bnicht nur\b[^.]{0,120}\bsondern\b'),
    'buzzwords_en': re.compile(r'(?i)\b(?:delve|tapestry|testament|pivotal|intricate|multifaceted|foster|realm|underscore|showcase|seamless|leverage|vibrant|nuanced|navigat\w*|landscape)\b'),
    'transitions_en': re.compile(r"(?i)\b(?:moreover|furthermore|additionally|in conclusion|ultimately|it is important to note|it's worth noting)\b"),
    'transitions_de': re.compile(r'(?i)\b(?:darüber hinaus|zudem|insgesamt|letztendlich|es ist wichtig)\b'),
    'lowercase_i': re.compile(r'\bi\b'),
}
LLM_ADDS = re.compile(r'(?i)\b(?:furthermore|crucial|potential|significant|challenges|individuals|essential|ultimately|understanding|approach|regarding|despite|moreover|müssen|sondern|unsere|herausforderung\w*|gesellschaft|verantwortung|sicherstellen|lösungen|zukunft|gleichzeitig|unterstützung)\b|—|[“”„]')
LLM_REMOVES = re.compile(r"(?i)\b(?:i|you|ich|ja|jetzt|dann|halt|mal|man|noch)\b|'")
UPPER = re.compile(r'[A-ZÄÖÜ]')
DIGIT = re.compile(r'\d')
PUNCT = re.compile(r'[^\w\s]')

def stylo(t):
    words = t.split()
    n = max(len(words), 1)
    sl = np.array([len(s.split()) for s in SENT.split(t) if s.strip()] or [n])
    f = {
        'log_words': np.log1p(n),
        'log_chars': np.log1p(len(t)),
        'avg_word_len': len(t) / n,
        'sent_len_mean': sl.mean(),
        'sent_len_cv': sl.std() / max(sl.mean(), 1),
        'log_sents': np.log1p(len(sl)),
        'ttr_200': len({w.lower() for w in words[:200]}) / min(n, 200),
        'punct': len(PUNCT.findall(t)) / max(len(t), 1),
        'upper': len(UPPER.findall(t)) / max(len(t), 1),
        'digit': len(DIGIT.findall(t)) / max(len(t), 1),
        **{f'm_{k}': len(p.findall(t)) / n * 1000 for k, p in MARKERS.items()},
    }
    quarters = [' '.join(q) for q in np.array_split(np.array(words, dtype=object), 4)]
    qn = [max(len(q.split()), 1) for q in quarters]
    for name, p in [('adds', LLM_ADDS), ('removes', LLM_REMOVES)]:
        d = np.array([len(p.findall(q)) / k for q, k in zip(quarters, qn)])
        f.update({f'{name}_q{i}': x for i, x in enumerate(d)})
        f[f'{name}_std'] = d.std()
        f[f'{name}_tail_minus_head'] = d[2:].mean() - d[:2].mean()
    wl = np.array([len(q) / k for q, k in zip(quarters, qn)])
    f['wordlen_tail_minus_head'] = wl[2:].mean() - wl[:2].mean()
    return f

texts = pd.concat([tr, sub_tr, va, test])[['id', 'text']].drop_duplicates('id')
F = pd.DataFrame(Parallel(n_jobs=-1, batch_size=256)(delayed(stylo)(t) for t in texts.text), index=texts.id)
del texts
gc.collect()
F.shape

(179535, 36)

In [3]:
F.loc[va.id].groupby(va.label.values).median().T.round(3)

,ai,fusion,human
log_words,6.148,6.094,6.178
log_chars,8.047,7.920,7.971
avg_word_len,6.606,6.075,5.861
sent_len_mean,18.000,17.250,17.409
sent_len_cv,0.438,0.575,0.617
log_sents,3.296,3.258,3.332
ttr_200,0.705,0.665,0.650
punct,0.025,0.023,0.025
upper,0.021,0.025,0.030
digit,0.000,0.003,0.004


## 1. Stylometry only

In [4]:
hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=.1, early_stopping=True, random_state=SEED)
hgb.fit(F.loc[tr.id], tr.label)

def save_scores(name, classes, val_scores, test_scores):
    for split, df, s in [('val', va, val_scores), ('test', test, test_scores)]:
        pd.DataFrame(s, columns=classes, index=df.id).to_csv(f'{split}_{name}.csv')

va_p, te_p = hgb.predict_proba(F.loc[va.id]), hgb.predict_proba(F.loc[test.id])
save_scores('probs_stylo', hgb.classes_, np.log(va_p + 1e-9), np.log(te_p + 1e-9))
preds = {'stylometry_only': pd.Series(hgb.classes_[va_p.argmax(1)], index=va.id)}
round(f1_score(va.label, preds['stylometry_only'].values, average='macro'), 4)

0.798

## 2. TF-IDF + stylometry
Baseline vectorizers and model config (`baseline_best.json` from the baseline notebook). Stylometric features are standardized and scaled by `alpha`; `alpha` is chosen on the 50k subsample, `alpha = 0` reproduces the baseline.

In [5]:
VECS = {
    'word': TfidfVectorizer(ngram_range=(1, 2), token_pattern=r'(?u)\w+|[^\w\s]', min_df=3, max_features=200_000, sublinear_tf=True, dtype=np.float32),
    'char': TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, max_features=200_000, sublinear_tf=True, dtype=np.float32),
}
KINDS = BASE['feats'].split('+')

def transform(vec, texts, chunk=10_000):
    return sp.vstack([vec.transform(texts.iloc[i:i + chunk]) for i in range(0, len(texts), chunk)]).tocsr()

for k in KINDS:
    VECS[k].fit(sub_tr.text)
scaler = StandardScaler().fit(F.loc[sub_tr.id])

def tfidf(df):
    return sp.hstack([transform(VECS[k], df.text) for k in KINDS]).tocsr()

def stylo_matrix(df):
    return sp.csr_matrix(np.nan_to_num(scaler.transform(F.loc[df.id])).astype(np.float32))

def join(T, S, alpha):
    return sp.hstack([T, S * alpha]).tocsr()

def make_model():
    cw = None if BASE['class_weight'] == 'None' else 'balanced'
    if BASE['model'] == 'svc':
        return LinearSVC(C=BASE['C'], class_weight=cw)
    return LogisticRegression(C=BASE['C'], class_weight=cw, max_iter=2000)

T_sub, S_sub, T_va, S_va = tfidf(sub_tr), stylo_matrix(sub_tr), tfidf(va), stylo_matrix(va)
alpha_grid = pd.Series({
    a: f1_score(va.label, make_model().fit(join(T_sub, S_sub, a), sub_tr.label).predict(join(T_va, S_va, a)), average='macro')
    for a in [0, .1, .3, 1, 3]
}, name='macro_f1').round(4)
ALPHA = alpha_grid.idxmax()
alpha_grid

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.0    0.8680
0.1    0.8713
0.3    0.8709
1.0    0.8710
3.0    0.8714
Name: macro_f1, dtype: float64

In [6]:
del T_sub, S_sub
gc.collect()
X_tr, X_va, X_te = join(tfidf(tr), stylo_matrix(tr), ALPHA), join(T_va, S_va, ALPHA), join(tfidf(test), stylo_matrix(test), ALPHA)
model = make_model().fit(X_tr, tr.label)
save_scores('scores_tfidf_stylo', model.classes_, model.decision_function(X_va), model.decision_function(X_te))
preds['tfidf_stylo'] = pd.Series(model.predict(X_va), index=va.id)
ALPHA

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


np.float64(3.0)

## Comparison on the same val fold

In [7]:
base_path = next(INPUT.rglob('val_scores_baseline.csv'), None)
if base_path:
    preds = {'baseline': pd.read_csv(base_path, index_col='id').loc[va.id].idxmax(axis=1), **preds}

y = va.set_index('id').label
def scores(p):
    f = f1_score(y, p.loc[y.index], labels=LABELS, average=None)
    return {**dict(zip(LABELS, f)), 'macro': f.mean()}

pd.DataFrame({k: scores(p) for k, p in preds.items()}).T.round(4)

,human,ai,fusion,macro
baseline,0.8425,0.9888,0.8493,0.8935
stylometry_only,0.7256,0.9448,0.7236,0.7980
tfidf_stylo,0.8334,0.9879,0.8408,0.8874


In [8]:
v = va.set_index('id')
pd.DataFrame({
    k: pd.concat({by: v.assign(pred=p.loc[v.index]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
                  for by in ['lang', 'domain']})
    for k, p in preds.items()
}).round(4)

/tmp/ipykernel_16/1356942826.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  k: pd.concat({by: v.assign(pred=p.loc[v.index]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
/tmp/ipykernel_16/1356942826.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  k: pd.concat({by: v.assign(pred=p.loc[v.index]).groupby(by).apply(lambda d: f1_score(d.label, d.pred, average='macro'))
/tmp/ipykernel_16/

baseline  stylometry_only  tfidf_stylo
lang   de-DE                     0.9205           0.8018       0.9145
       en-EN                     0.8867           0.7969       0.8804
domain arxiv_papers              0.8918           0.7927       0.8477
       blog_authorship_corpus    0.7768           0.6575       0.7653
       bundestag                 0.9206           0.8027       0.9148
       cnn_news                  0.9190           0.7889       0.9215
       euro_court_cases          0.9260           0.8251       0.9161
       gutenberg                 0.7746           0.6324       0.6822
       house_of_commons          0.8983           0.8089       0.8883
       student_essays            0.9056           0.8540       0.9027